In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import xgboost as xgb
import joblib
import os
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

project_path = '/Workspace/Users/rohanoruganti123@gmail.com/larry/'

model_path = os.path.join(project_path, 'upi_model.json')
encoder_path = os.path.join(project_path, 'encoders.pkl')
metadata_path = os.path.join(project_path, 'metadata.pkl')
test_file = os.path.join(project_path, 'testupi.csv')
test_file = os.path.join(volume_path, 'testupi.csv') # Using the full test set

# 2. Load Artifacts
print("Loading model and metadata...")
model = xgb.XGBClassifier()
model.load_model(model_path)
encoders = joblib.load(encoder_path)
metadata = joblib.load(metadata_path)
threshold = metadata['threshold']

# 3. Load and Preprocess Test Data
df_test = pd.read_csv(test_file)

# Apply the SAME encoding as training
for col, le in encoders.items():
    # Handle unseen categories in test data by mapping to 'Unknown' if necessary
    df_test[col] = df_test[col].astype(str).fillna('Unknown')
    # Map classes, handle potential new labels not seen in training
    df_test[col] = df_test[col].map(lambda s: s if s in le.classes_ else le.classes_[0])
    df_test[col] = le.transform(df_test[col])

X_test = df_test.drop(['transaction id', 'timestamp', 'fraud_flag', 'transaction_status'], axis=1)
y_test = df_test['fraud_flag']

# 4. Predict
print(f"Predicting with threshold: {threshold}...")
y_probs = model.predict_proba(X_test)[:, 1]
y_pred = (y_probs >= threshold).astype(int)

# 5. RESULTS
accuracy = accuracy_score(y_test, y_pred)
print("-" * 30)
print(f"CONFIRMED ACCURACY: {accuracy:.4%}")
print("-" * 30)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))